# Objetivo
Generar samples y obtener una idea del escalado temporal, todo aprovechando de usar tecnicas de batching programadas; de esta manera generando datos para los modelos de ML clasificativos.

In [ ]:
import numpy as np
import pandas as pd
import time
import psutil
import pickle
from scipy.stats import qmc
from lib.oracle import OracleExecutor  # assumes your OracleExecutor is in oracle_wrapper.py


epsilon = 0.1
vev = 246

# Define the ranges for each parameter:
# [m_phi, m_A, sin_ba, tan_beta, lambda6, lambda7, m12_2]
# TODO
# 300 done 7 batches
# 290 
# 280 m_phi_base, m12_2_base = 280, 7.83990900e+00

m_phi_base, m12_2_base = 270, 7.83990900e+00

param_bounds = np.array([
    [m_phi_base - 3* epsilon, m_phi_base + 3*epsilon],    # m_phi (GeV)
    [300 - epsilon, 300 + epsilon],    # m_A   (GeV)
    [1.0 - epsilon, 1.0],        # sin(b - a)
    [10000.0 - epsilon, 10000.0 + epsilon],        # tan(beta)
    [0.1 - epsilon, 0.1 + epsilon],# lambda6
    [0, epsilon],# lambda7
    [m12_2_base - epsilon, m12_2_base + epsilon]  # m12^2, agregado para que sea pequeño
])

def generate_params(batch_idx, param_bounds, batch_size):
    sampler = qmc.LatinHypercube(d=7, seed=batch_idx)
    unit_sample = sampler.random(n=batch_size)
    param_list = qmc.scale(unit_sample, param_bounds[:,0], param_bounds[:,1])

    # fijando parametros
    param_list[:, 2] = 1 # sin(b-a)
    param_list[:, 3] = 10000 # tan_beta
    param_list[:, 4] = 0.1 # lambda6
    param_list[:, 5] = 0.0 # lambda7

    return param_list

# Prepare executor
executor = OracleExecutor(nthreads=4)

# Storage for performance metrics
perf_records = []





# Runs

In [2]:
import os
import glob
import pickle
import psutil
import time
import numpy as np
from scipy.stats import qmc

batch_size = 15_000
outdir = "data_batches"
max_merge_size_mb = 30

os.makedirs(outdir, exist_ok=True)

In [3]:
existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
batch_idx

16

In [4]:
for jth_run in range(3):
    # ------------------------
    # Determine next batch index
    # ------------------------
    existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
    batch_idx = len(existing_batches) + 1
    print(batch_idx)

    # ------------------------
    # Generate Latin-Hypercube sample
    # ------------------------
    param_list = generate_params(batch_idx, param_bounds, batch_size)

    # ------------------------
    # Run your oracle / executor
    # ------------------------
    t0 = time.perf_counter()
    results = executor.map(param_list.tolist(), use_threads=True)
    t1 = time.perf_counter()

    # ------------------------
    # Save this batch
    # ------------------------
    outfile = f"{outdir}/batch_{batch_idx}.pkl"
    with open(outfile, "wb") as f:
        pickle.dump({"params": param_list, "results": results}, f)

    print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")



16
Batch 16 saved (15000 points) in 415.7s → data_batches/batch_16.pkl
17
Batch 17 saved (15000 points) in 415.0s → data_batches/batch_17.pkl
18
Batch 18 saved (15000 points) in 433.6s → data_batches/batch_18.pkl
19
Batch 19 saved (15000 points) in 412.4s → data_batches/batch_19.pkl
20
Batch 20 saved (15000 points) in 437.9s → data_batches/batch_20.pkl


In [ ]:
for jth_run in range(4):
    # ------------------------
    # Determine next batch index
    # ------------------------
    existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
    batch_idx = len(existing_batches) + 1
    print(batch_idx)

    # ------------------------
    # Generate Latin-Hypercube sample
    # ------------------------
    param_list = generate_params(batch_idx, param_bounds, batch_size)

    # ------------------------
    # Run your oracle / executor
    # ------------------------
    t0 = time.perf_counter()
    results = executor.map(param_list.tolist(), use_threads=True)
    t1 = time.perf_counter()

    # ------------------------
    # Save this batch
    # ------------------------
    outfile = f"{outdir}/batch_{batch_idx}.pkl"
    with open(outfile, "wb") as f:
        pickle.dump({"params": param_list, "results": results}, f)

    print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")



16


In [4]:
# ------------------------
# Determine next batch index
# ------------------------
existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
print(batch_idx)

# ------------------------
# Generate Latin-Hypercube sample
# ------------------------
sampler = qmc.LatinHypercube(d=7, seed=batch_idx)
unit_sample = sampler.random(n=batch_size)
param_list = qmc.scale(unit_sample, param_bounds[:,0], param_bounds[:,1])

# ------------------------
# Run your oracle / executor
# ------------------------
t0 = time.perf_counter()
results = executor.map(param_list.tolist(), use_threads=True)
t1 = time.perf_counter()

# ------------------------
# Save this batch
# ------------------------
outfile = f"{outdir}/batch_{batch_idx}.pkl"
with open(outfile, "wb") as f:
    pickle.dump({"params": param_list, "results": results}, f)

print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")



7
Batch 7 saved (15000 points) in 392.2s → data_batches/batch_7.pkl


In [16]:
# ------------------------
# Determine next batch index
# ------------------------
existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
print(batch_idx)

# ------------------------
# Generate Latin-Hypercube sample
# ------------------------
sampler = qmc.LatinHypercube(d=7, seed=batch_idx)
unit_sample = sampler.random(n=batch_size)
param_list = qmc.scale(unit_sample, param_bounds[:,0], param_bounds[:,1])

# ------------------------
# Run your oracle / executor
# ------------------------
t0 = time.perf_counter()
results = executor.map(param_list.tolist(), use_threads=False)
t1 = time.perf_counter()

# ------------------------
# Save this batch
# ------------------------
outfile = f"{outdir}/batch_{batch_idx}.pkl"
with open(outfile, "wb") as f:
    pickle.dump({"params": param_list, "results": results}, f)

print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")



5
Batch 5 saved (15000 points) in 863.3s → data_batches/batch_5.pkl


In [14]:
# ------------------------
# Determine next batch index
# ------------------------
existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
print(batch_idx)

# ------------------------
# Generate Latin-Hypercube sample
# ------------------------
sampler = qmc.LatinHypercube(d=7, seed=batch_idx)
unit_sample = sampler.random(n=batch_size)
param_list = qmc.scale(unit_sample, param_bounds[:,0], param_bounds[:,1])

# ------------------------
# Run your oracle / executor
# ------------------------
t0 = time.perf_counter()
results = executor.map(param_list.tolist(), use_threads=False)
t1 = time.perf_counter()

# ------------------------
# Save this batch
# ------------------------
outfile = f"{outdir}/batch_{batch_idx}.pkl"
with open(outfile, "wb") as f:
    pickle.dump({"params": param_list, "results": results}, f)

print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")



3
Batch 3 saved (15000 points) in 900.0s → data_batches/batch_3.pkl



# Testing Speed

In [ ]:
# Define the sampling sizes
sample_sizes = [1, 10, 100, 1_000, 10_000]

for n in sample_sizes:
    # Generate Latin Hypercube samples in [0,1]^7, then scale
    sampler = qmc.LatinHypercube(d=7)
    sample_unit = sampler.random(n)
    param_list = qmc.scale(sample_unit, param_bounds[:,0], param_bounds[:,1])
    
    # Measure memory before run
    process = psutil.Process()
    mem_before = process.memory_info().rss
    
    # Run and time
    t0 = time.perf_counter()
    results = executor.map(param_list.tolist(), use_threads=False)
    t1 = time.perf_counter()
    
    mem_after = process.memory_info().rss
    delta_mem = (mem_after - mem_before) / (1024**2)  # in MB
    
    # Save raw results for this batch
    with open(f"oracle_results_{n}.pkl", "wb") as f:
        pickle.dump(results, f)
    
    # Record performance
    perf_records.append({
        "n_points": n,
        "time_sec": t1 - t0,
        "mem_delta_MB": delta_mem
    })
    print(f"Completed batch {n}: time={t1-t0:.2f}s, memory Δ={delta_mem:.1f}MB")

# Save performance table
df_perf = pd.DataFrame(perf_records)
df_perf.to_csv("performance_scaling.csv", index=False)



# Merging

In [ ]:
# ------------------------
# Merge old batches if they exceed size threshold
# ------------------------
def merge_batches(folder, batch_prefix="batch_", merged_prefix="merged_", max_size_mb=30):
    # Count existing merged files to avoid overwrite
    existing_merged = sorted(glob.glob(f"{folder}/{merged_prefix}*.pkl"))
    merge_idx = len(existing_merged) + 1
    
    # Only consider raw batch files
    batch_files = sorted(glob.glob(f"{folder}/{batch_prefix}*.pkl"))
    acc_size = 0
    group = []

    for fp in batch_files:
        fsize = os.path.getsize(fp)
        if (acc_size + fsize) / (1024**2) > max_size_mb and group:
            # Merge current group
            merged_data = []
            for gfp in group:
                with open(gfp, "rb") as gf:
                    merged_data.append(pickle.load(gf))
                os.remove(gfp)
            mout = f"{folder}/{merged_prefix}{merge_idx}.pkl"
            with open(mout, "wb") as mf:
                pickle.dump(merged_data, mf)
            print(f"Merged {len(group)} batches into {mout}")
            merge_idx += 1
            group, acc_size = [], 0

        group.append(fp)
        acc_size += fsize

    # Merge any remaining files
    if group:
        merged_data = []
        for gfp in group:
            with open(gfp, "rb") as gf:
                merged_data.append(pickle.load(gf))
            os.remove(gfp)
        mout = f"{folder}/{merged_prefix}{merge_idx}.pkl"
        with open(mout, "wb") as mf:
            pickle.dump(merged_data, mf)
        print(f"Merged {len(group)} batches into {mout}")

# Call merge
merge_batches(outdir, max_merge_size_mb=max_merge_size_mb)
""")

Merged 6 batches into data_batches/merged_1.pkl
Merged 6 batches into data_batches/merged_2.pkl
Merged 3 batches into data_batches/merged_3.pkl
